# Benchmark Models: Popularity-based Recommendation
**Mục tiêu:** Triển khai mô hình gợi ý theo độ phổ biến (Popularity-based) để thiết lập mức hiệu năng cơ bản (Baseline). 

**Logic:** Gợi ý cho người dùng những bộ phim có số lượng tương tác (Rating $\ge$ 4.0) cao nhất trong toàn bộ hệ thống. Đây là mô hình đơn giản nhất nhưng thường rất hiệu quả trong việc giải quyết vấn đề "khởi động lạnh".

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append('../../')
from src.utils import calculate_metrics

DATA_PROCESSED_PATH = "../../data/processed/"

df = pd.read_pickle(DATA_PROCESSED_PATH + 'processed_ratings.pkl')

train_data = df.sample(frac=0.8, random_state=42)
test_data = df.drop(train_data.index)

print(f"Số lượng mẫu huấn luyện: {len(train_data)}")
print(f"Số lượng mẫu kiểm thử: {len(test_data)}")

def get_popular_movies(train_df, top_k=10):
    popular_list = train_df.groupby('movieId').size().sort_values(ascending=False)
    return popular_list.head(top_k).index.tolist()

top_10_popular = get_popular_movies(train_data, top_k=10)
print(f"Top 10 phim phổ biến nhất: {top_10_popular}")

test_user_interactions = test_data.groupby('userId')['movieId'].apply(list).to_dict()

recalls = []
ndcgs = []

for user_id, actual_movies in test_user_interactions.items():
    predicted_movies = top_10_popular
    
    recall, ndcg = calculate_metrics(actual_movies, predicted_movies, k=10)
    recalls.append(recall)
    ndcgs.append(ndcg)

print(f"--- Kết quả Baseline (Popularity) ---")
print(f"Average Recall@10: {np.mean(recalls):.4f}")
print(f"Average nDCG@10: {np.mean(ndcgs):.4f}")